9. Aplikovanie masiek na FITS cutouty

Táto časť načíta všetky FITS cutouty a k nim prislúchajúce masky, prípadne masky interpoluje na rovnaké rozlíšenie, aplikuje ich na dáta a vynuluje (NaN) pixely mimo masky. Výsledkom sú FITS súbory obsahujúce iba oblasť galaxie.

In [ ]:
import os
import glob
import numpy as np
from astropy.io import fits
from PIL import Image
from skimage.transform import resize
from tqdm import tqdm

fits_dir  = "FITS_VIS_LOC_EXTRACTED"
masks_dir = "MASKS_ALL"
out_dir   = "FITS_CUTOUTS_NEW"
os.makedirs(out_dir, exist_ok=True)

fits_files = sorted(glob.glob(os.path.join(fits_dir, "*.fits")))
if not fits_files:
    raise SystemExit(f"V priečinku {fits_dir} nie sú žiadne FITS súbory.")

print(f"Nájdených FITS súborov: {len(fits_files)}")

def find_corresponding_mask(key):
    """
    Nájde masku podľa kľúča (názvu bez .fits).
    Masky môžu mať rôzne prípony (jpg, jpg.jpg, png).
    """
    candidates = [
        os.path.join(masks_dir, key + ".jpg"),
        os.path.join(masks_dir, key + ".jpg.jpg"),
        os.path.join(masks_dir, key + ".png"),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return None

for fits_path in tqdm(fits_files, desc="Spracovávam FITS"):

    base = os.path.basename(fits_path)
    key = base.replace(".fits", "")  # napr. XYZ.jpg.fits → key = XYZ.jpg

    mask_path = find_corresponding_mask(key)
    if mask_path is None:
        print(f"\n!!! Maska nenájdená pre FITS: {base}")
        continue

    # Načítanie FITS dát
    with fits.open(fits_path) as hdul:
        data = np.array(hdul[0].data, dtype=float)
        header = hdul[0].header

    # Načítanie masky
    img = Image.open(mask_path).convert("L")
    mask = np.array(img)

    # Prispôsobenie rozlíšenia masky k dátam
    if mask.shape != data.shape:
        mask = resize(mask, data.shape, order=0, preserve_range=True).astype(np.uint8)

    # Otočenie FITS (ak je potrebné zjednotiť orientáciu)
    data = np.flipud(data)

    # Aplikovanie masky
    mask_bin = (mask > 128).astype(np.uint8)
    cutout = np.copy(data)
    cutout[mask_bin == 0] = np.nan

    out_path = os.path.join(out_dir, f"{key}_cutout.fits")
    fits.PrimaryHDU(data=cutout, header=header).writeto(out_path, overwrite=True)

print("\nHotovo! Všetky dostupné FITS boli spracované.")

10. Odhad pozadia pomocou sigma‑clip

Táto časť odhaduje strednú hodnotu a rozptyl pozadia na reprezentatívnom FITS súbore pomocou náhodného vzorkovania boxov a sigma‑clippingu. Výsledné hodnoty sa používajú pri odstraňovaní pozadia.

In [ ]:
import numpy as np
from astropy.io import fits
from astropy.stats import sigma_clip

# Načítanie reprezentatívneho FITS súboru
data = fits.getdata("FITS_vis_loc/EUC_MER_BGSUB-MOSAIC-VIS_TILE102042913-EE5E4F_20241021T041311.751862Z_00.00.fits")

box = 100      # veľkosť boxu v pixeloch
n_boxes = 50   # počet náhodných boxov

means = []
stds  = []

ny, nx = data.shape
rng = np.random.default_rng(1234)

for _ in range(n_boxes):
    y = rng.integers(0, ny - box)
    x = rng.integers(0, nx - box)

    cut = data[y:y+box, x:x+box]

    # Sigma-clipping na odstránenie outlierov
    clipped = sigma_clip(cut, sigma=3)

    means.append(np.nanmean(clipped))
    stds.append(np.nanstd(clipped))

bg_mean = np.mean(means)
bg_std  = np.mean(stds)

print("Background mean:", bg_mean)
print("Background std :", bg_std)

11. Odstránenie pozadia z cutoutov

Táto časť načíta všetky maskované FITS cutouty, použije globálny odhad pozadia a adaptívny sigma‑threshold podľa veľkosti galaxie. Pixely pod prahom sú považované za pozadie a nastavené na NaN.

In [ ]:
import os
import glob
import numpy as np
from astropy.io import fits
from tqdm import tqdm

fits_dir = "FITS_CUTOUTS_NEW"
out_dir  = "CUTOUT_NEW_FINALS_"
os.makedirs(out_dir, exist_ok=True)

# Hodnoty pozadia získané zo sigma-clip analýzy
BG_MEAN = 0.00031286004
BG_STD  = 0.0029045846

fits_files = sorted(glob.glob(os.path.join(fits_dir, "*.fits")))
if not fits_files:
    raise SystemExit(f"V priečinku {fits_dir} nie sú žiadne FITS súbory.")

print(f"Spracovávam {len(fits_files)} FITS súborov...")

for fits_path in tqdm(fits_files, desc="Spracovanie FITS"):
    key = os.path.basename(fits_path).replace(".fits", "")

    with fits.open(fits_path) as hdul:
        img = np.array(hdul[0].data, dtype=float)
        header = hdul[0].header

    # Otočenie, ak je potrebné zjednotiť orientáciu
    img = np.flipud(img)

    # Odhad veľkosti galaxie (max. rozmer cutoutu)
    gal_size = max(img.shape)

    # Adaptívny sigma threshold podľa veľkosti
    if gal_size <= 60:
        sigma = 3
    else:
        sigma = 5

    BACKGROUND_LIMIT = BG_MEAN + sigma * BG_STD

    # Odstránenie pozadia
    cutout_mask = img > BACKGROUND_LIMIT
    cutout = np.copy(img)
    cutout[~cutout_mask] = np.nan

    out_path = os.path.join(out_dir, f"{key}_cutout.fits")
    fits.PrimaryHDU(data=cutout, header=header).writeto(out_path, overwrite=True)

print("\nHotovo. Pozadie bolo odstránené z všetkých cutoutov.")

12. Vizualizácia warpu a PCA zarovnanie diskov

Táto časť náhodne vyberie podmnožinu FITS cutoutov, pre každý objekt odhadne orientáciu disku pomocou PCA, zarovná ho do horizontálnej polohy, vykoná binning v súradniciach a vypočíta „flux‑weighted“ stred profilu v závislosti od polohy v disku. Výsledkom je vizualizácia rotovaného disku s prekreslenou krivkou warpu.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import random
from astropy.io import fits
import matplotlib.colors as colors

# Vstupný priečinok s FITS cutoutmi
input_dir = "FITS_CUTOUTS_NEW"

# Počet náhodne vybraných galaxií na zobrazenie
N_SHOW = 30

# Veľkosť binu v rotovaných súradniciach
BIN_SIZE = 2

# ======= ZÍSKANIE ZOZNAMU FITS SÚBOROV =======
all_fits_files = sorted([f for f in os.listdir(input_dir) if f.endswith(".fits")])

if len(all_fits_files) < N_SHOW:
    raise ValueError(f"Not enough FITS files in {input_dir} to select {N_SHOW} files.")

# ======= NÁHODNÝ VÝBER FITS SÚBOROV =======
fits_files = random.sample(all_fits_files, N_SHOW)
fits_paths = [os.path.join(input_dir, f) for f in fits_files]

def rotate_point(x, y, ang):
    """Rotácia bodu (x, y) o uhol ang (v radiánoch)."""
    cos_t = np.cos(ang)
    sin_t = np.sin(ang)
    return x * cos_t - y * sin_t, x * sin_t + y * cos_t

def calculate_flux_center(dat, x_value):
    """
    Vypočíta fluxom vážený stred v osi Y pre daný X_bin.
    Integruje pozdĺž Y pomocou jednoduchého numerického prístupu.
    """
    subset = dat[dat['X_bin'] == x_value].copy().reset_index(drop=True)

    if len(subset) < 2:
        return None

    subset = subset.sort_values('Y_rot')

    numerator = 0.0
    denominator = 0.0

    for i in range(1, len(subset)):
        dy    = subset['Y_rot'].iloc[i] - subset['Y_rot'].iloc[i-1]
        flux  = subset['value'].iloc[i]
        y_pos = subset['Y_rot'].iloc[i]

        numerator   += flux * y_pos * dy
        denominator += flux * dy

    if denominator == 0:
        return None

    return numerator / denominator

# ======= PRÍPRAVA MREŽKY SUBPLOTOV =======
rows = int(np.ceil(N_SHOW / 2))
cols = 2

fig, axes = plt.subplots(rows, cols, figsize=(14, 4*rows))
axes = axes.flatten()

for i, file in enumerate(fits_paths):

    ax = axes[i]

    # ======= NAČÍTANIE FITS =======
    with fits.open(file) as hdu:
        img = hdu[0].data

    # Konverzia na float64 (riešenie endian/typových problémov)
    img = np.array(img, dtype=np.float64)

    # Výber nenan a nenulových pixelov
    y_idx, x_idx = np.where(~np.isnan(img) & (img != 0))
    values = img[y_idx, x_idx]

    data = pd.DataFrame({
        'X': x_idx,
        'Y': y_idx,
        'value': values
    })

    # ======= ODSTRÁNENIE NEGATÍVNEHO FLUXU =======
    weights = np.clip(data['value'].values, 0, None)

    # ======= ODHAD JADRA GALAXIE (TOP 1 % NAJJAŠTEJŠÍCH PIXELOV) =======
    threshold = np.percentile(data['value'], 99)
    core_pixels = data[data['value'] >= threshold]

    center_x = np.average(core_pixels['X'], weights=core_pixels['value'])
    center_y = np.average(core_pixels['Y'], weights=core_pixels['value'])

    # ======= POSUN SÚRADNÍC DO STREDU GALAXIE =======
    data['X'] -= center_x
    data['Y'] -= center_y

    # ======= PCA NA ODHAD HLAVNEJ OSI DISKU =======
    coords = data[['X','Y']].values

    mean = np.average(coords, axis=0, weights=weights)
    coords_centered = coords - mean

    cov = np.cov(coords_centered.T, aweights=weights)
    eigvals, eigvecs = np.linalg.eigh(cov)

    major_axis = eigvecs[:, np.argmax(eigvals)]
    angle = np.arctan2(major_axis[1], major_axis[0])

    # ======= ROTÁCIA DO RÁMCA S DISKOM V HORIZONTE =======
    data_rot = data.copy()
    data_rot['X_rot'], data_rot['Y_rot'] = zip(*data.apply(
        lambda row: rotate_point(row['X'], row['Y'], -angle),
        axis=1
    ))

    # ======= OPAKOVANÝ ODHAD JADRA V ROTOVANOM RÁMCI =======
    flux_threshold = np.percentile(data_rot['value'], 95)
    core_pixels = data_rot[data_rot['value'] >= flux_threshold]
    core_y = np.average(core_pixels['Y_rot'], weights=core_pixels['value'])

    # Posun v osi Y tak, aby jadro ležalo blízko Y=0
    data_rot['Y_rot'] -= core_y

    # ======= BINNING V ROTOVANÝCH SÚRADNICIACH =======
    data_rot['X_bin'] = (np.floor(data_rot['X_rot'] / BIN_SIZE) * BIN_SIZE).astype(int)
    data_rot['Y_bin'] = (np.floor(data_rot['Y_rot'] / BIN_SIZE) * BIN_SIZE).astype(int)

    binned_data = data_rot.groupby(['X_bin','Y_bin']).agg({
        'X_rot':'mean',
        'Y_rot':'mean',
        'value':'mean'
    }).reset_index()

    binned_left  = binned_data[binned_data['X_bin'] <= 0].copy()
    binned_right = binned_data[binned_data['X_bin'] >= 0].copy()

    # ======= VÝPOČET WARPU NA ĽAVEJ A PRAVEJ STRANE =======
    x_vals_left  = binned_left['X_bin'].drop_duplicates().sort_values()
    x_vals_right = binned_right['X_bin'].drop_duplicates().sort_values()

    warp_left = [{'X': x, 'Y_center': calculate_flux_center(binned_left, x)} for x in x_vals_left]
    warp_right = [{'X': x, 'Y_center': calculate_flux_center(binned_right, x)} for x in x_vals_right]

    warp_left  = pd.DataFrame([w for w in warp_left  if w['Y_center'] is not None])
    warp_right = pd.DataFrame([w for w in warp_right if w['Y_center'] is not None])

    warp_combined = pd.concat([warp_left, warp_right]).sort_values('X').reset_index(drop=True)

    # ======= ZAROVNANIE CENTRÁLNEJ ČASTI PROFILU NA Y≈0 =======
    if len(warp_combined) > 0:
        center_region = warp_combined[np.abs(warp_combined['X']) < 10]
        if len(center_region) > 0:
            y0 = np.average(center_region['Y_center'])
            warp_combined['Y_center'] -= y0

    # ======= VYKRESLENIE ROTOVANÝCH DÁT A WARPU =======
    sc = ax.scatter(
        data_rot['X_rot'],
        data_rot['Y_rot'],
        c=data_rot['value'],
        norm=colors.LogNorm(vmin=1e-3, vmax=1),
        s=300,
        cmap='viridis'
    )

    if len(warp_combined) > 0:
        ax.plot(
            warp_combined['X'],
            warp_combined['Y_center'],
            'r-',
            linewidth=2
        )

    # Referenčná horizontálna os
    ax.axhline(0, color='white', linestyle='--', alpha=0.7)

    ax.set_title(os.path.basename(file), fontsize=10)

    # Dynamické nastavenie rozsahov s malou rezervou
    margin = 0.1
    x_min, x_max = data_rot['X_rot'].min(), data_rot['X_rot'].max()
    y_min, y_max = data_rot['Y_rot'].min(), data_rot['Y_rot'].max()

    x_range = x_max - x_min
    y_range = y_max - y_min

    ax.set_xlim(x_min - margin*x_range, x_max + margin*x_range)
    ax.set_ylim(y_min - margin*y_range, y_max + margin*y_range)

    ax.set_aspect('equal')
    ax.grid(alpha=0.3)

# ======= ODSTRÁNENIE NEPOUŽITÝCH SUBPLOTOV =======
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

13. Výpočet warpu, extrakcia metadát a archivácia FITS verzií

Táto časť pipeline spracúva všetky finálne FITS cutouty, vykonáva PCA zarovnanie disku, výpočet warpu na ľavej a pravej strane, meria rozsahy v rotovanom rámci a extrahuje súradnice RA/DEC z katalógu. Pre každý objekt sa vytvorí ZIP balík obsahujúci pôvodný FITS, cutout a sigma‑clipped verziu. Výsledky sa ukladajú do tabuľky warp_results_original_cutouts.csv.

In [ ]:
import numpy as np
import pandas as pd
import os
import shutil
import zipfile
from astropy.io import fits
from tqdm import tqdm
import warnings

# Potlačenie warningov, ktoré narúšajú výpis progress baru
warnings.filterwarnings("ignore")

# ======== Nastavenia =========
input_dir    = "FITS_CUTOUTS_NEW"      # finálne cutouty po aplikovaní masiek
original_dir = "FITS_CUTOUTS"          # pôvodné cutouty
sigma_dir    = "CUTOUT_NEW_FINALS_"    # cutouty po odstránení pozadia
BIN_SIZE     = 2                       # veľkosť binu v rotovanom rámci

# Katalóg s RA/DEC a názvami súborov
catalog = pd.read_csv("modified_dataset.csv")

# Zoznam FITS súborov na spracovanie
all_fits_files = sorted([f for f in os.listdir(input_dir) if f.endswith(".fits")])
fits_paths = [os.path.join(input_dir, f) for f in all_fits_files]

def rotate_points(X, Y, ang):
    """Rotácia bodov (X, Y) o uhol ang (radiány)."""
    cos_t, sin_t = np.cos(ang), np.sin(ang)
    X_rot = X * cos_t - Y * sin_t
    Y_rot = X * sin_t + Y * cos_t
    return X_rot, Y_rot

def calculate_flux_center_vectorized(X, Y, value):
    """
    Výpočet fluxom váženého stredu profilu v osi Y pre každý X_bin.
    Používa jednoduchú numerickú integráciu.
    """
    unique_x = np.unique(X)
    results = []

    for x in unique_x:
        mask = X == x
        y_vals = Y[mask]
        v_vals = value[mask]

        if len(y_vals) < 2:
            continue

        sort_idx = np.argsort(y_vals)
        y_sorted = y_vals[sort_idx]
        v_sorted = v_vals[sort_idx]

        dy = np.diff(y_sorted)
        if np.sum(dy * v_sorted[1:]) == 0:
            continue

        numerator   = np.sum(v_sorted[1:] * y_sorted[1:] * dy)
        denominator = np.sum(v_sorted[1:] * dy)

        results.append((x, numerator / denominator))

    return np.array(results)

results = []

# Priečinky pre ZIP archívy
output_zip_folder = "output_zips"
os.makedirs(output_zip_folder, exist_ok=True)

tmp_zip_folder = "tmp_zip_folder"
os.makedirs(tmp_zip_folder, exist_ok=True)

# ======== Hlavná slučka spracovania FITS súborov =========
for file in tqdm(
        fits_paths,
        desc="Processing FITS files",
        dynamic_ncols=True,
        leave=False
    ):

    file_base = os.path.basename(file).split('.')[0]

    # ======== Načítanie FITS =========
    with fits.open(file) as hdu:
        img = np.array(hdu[0].data, dtype=np.float64)

    # Výber nenan a nenulových pixelov
    mask = (~np.isnan(img)) & (img != 0)
    y_idx, x_idx = np.nonzero(mask)
    values = img[y_idx, x_idx]

    # ======== Ošetrenie prázdnych FITS =========
    if len(values) == 0:
        results.append({
            "galaxy_id": file_base,
            "left_side_integral": np.nan,
            "avg_left_offset": np.nan,
            "right_side_integral": np.nan,
            "avg_right_offset": np.nan,
            "left_edge": np.nan,
            "right_edge": np.nan,
            "x_span": np.nan,
            "y_span": np.nan,
            "RA": np.nan,
            "DEC": np.nan,
            "zip_file": None
        })
        continue

    # ======== Príprava dátového rámca =========
    data = pd.DataFrame({'X': x_idx, 'Y': y_idx, 'value': values})
    weights = np.clip(values, 0, None)

    # ======== Odhad jadra galaxie (top 1 % pixelov) =========
    threshold = np.percentile(values, 99)
    core_pixels = data[values >= threshold]

    center_x = np.average(core_pixels['X'], weights=core_pixels['value'])
    center_y = np.average(core_pixels['Y'], weights=core_pixels['value'])

    # Posun súradníc do stredu galaxie
    data['X'] -= center_x
    data['Y'] -= center_y

    # ======== PCA na odhad hlavnej osi disku =========
    coords = data[['X','Y']].values
    mean = np.average(coords, axis=0, weights=weights)
    coords_centered = coords - mean

    cov = np.cov(coords_centered.T, aweights=weights)
    eigvals, eigvecs = np.linalg.eigh(cov)

    major_axis = eigvecs[:, np.argmax(eigvals)]
    angle = np.arctan2(major_axis[1], major_axis[0])

    # ======== Rotácia do horizontálneho rámca =========
    data['X_rot'], data['Y_rot'] = rotate_points(
        data['X'].values,
        data['Y'].values,
        -angle
    )

    # ======== Opätovné zarovnanie jadra v osi Y =========
    flux_threshold = np.percentile(data['value'], 95)
    core_mask = data['value'] >= flux_threshold
    core_y = np.average(data['Y_rot'][core_mask], weights=data['value'][core_mask])

    data['Y_rot'] -= core_y

    # ======== Binning =========
    data['X_bin'] = (np.floor(data['X_rot'] / BIN_SIZE) * BIN_SIZE).astype(int)
    data['Y_bin'] = (np.floor(data['Y_rot'] / BIN_SIZE) * BIN_SIZE).astype(int)

    binned_data = data.groupby(['X_bin','Y_bin'], as_index=False).agg({
        'X_rot':'mean',
        'Y_rot':'mean',
        'value':'mean'
    })

    binned_left  = binned_data[binned_data['X_bin'] <= 0]
    binned_right = binned_data[binned_data['X_bin'] >= 0]

    # ======== Výpočet warpu =========
    warp_left_arr  = calculate_flux_center_vectorized(
        binned_left['X_bin'].values,
        binned_left['Y_rot'].values,
        binned_left['value'].values
    )

    warp_right_arr = calculate_flux_center_vectorized(
        binned_right['X_bin'].values,
        binned_right['Y_rot'].values,
        binned_right['value'].values
    )

    # Kombinácia ľavej a pravej strany
    if len(warp_left_arr) > 0 or len(warp_right_arr) > 0:
        warp_combined_arr = np.vstack([warp_left_arr, warp_right_arr])
    else:
        warp_combined_arr = np.empty((0,2))

    # Zarovnanie centrálnej oblasti na Y≈0
    if warp_combined_arr.size > 0:
        center_mask = np.abs(warp_combined_arr[:,0]) < 10
        if np.any(center_mask):
            y0 = np.mean(warp_combined_arr[center_mask,1])
            warp_combined_arr[:,1] -= y0

    # ======== Výpočet metrík warpu =========
    left_integral  = np.trapz(warp_left_arr[:,1],  warp_left_arr[:,0])  if len(warp_left_arr)  > 0 else np.nan
    right_integral = np.trapz(warp_right_arr[:,1], warp_right_arr[:,0]) if len(warp_right_arr) > 0 else np.nan

    avg_left_offset  = np.mean(warp_left_arr[:,1])  if len(warp_left_arr)  > 0 else np.nan
    avg_right_offset = np.mean(warp_right_arr[:,1]) if len(warp_right_arr) > 0 else np.nan

    left_edge  = warp_left_arr[0,1]   if len(warp_left_arr)  > 0 else np.nan
    right_edge = warp_right_arr[-1,1] if len(warp_right_arr) > 0 else np.nan

    x_span = data['X_rot'].max() - data['X_rot'].min()
    y_span = data['Y_rot'].max() - data['Y_rot'].min()

    # ======== RA/DEC z katalógu =========
    row = catalog[catalog['jpg_loc_gz_arcsinh_vis_only'].str.split('.').str[0] == file_base]
    ra  = row.iloc[0]['right_ascension'] if len(row) > 0 else np.nan
    dec = row.iloc[0]['declination']     if len(row) > 0 else np.nan

    # ======== Vytvorenie ZIP archívu =========
    for key, path in {"original":original_dir, "cutout":input_dir, "sigma":sigma_dir}.items():
        src = os.path.join(path, file_base + ".fits")
        if os.path.exists(src):
            shutil.copy(src, os.path.join(tmp_zip_folder, f"{file_base}_{key}.fits"))

    zip_name = os.path.join(output_zip_folder, f"{file_base}_all_versions.zip")

    with zipfile.ZipFile(zip_name, 'w') as zf:
        for f in os.listdir(tmp_zip_folder):
            if f.startswith(file_base):
                zf.write(os.path.join(tmp_zip_folder, f), arcname=f)

    # Reset dočasného priečinka
    shutil.rmtree(tmp_zip_folder)
    os.makedirs(tmp_zip_folder, exist_ok=True)

    # ======== Uloženie výsledkov =========
    results.append({
        "galaxy_id": file_base,
        "left_side_integral": left_integral,
        "avg_left_offset": avg_left_offset,
        "right_side_integral": right_integral,
        "avg_right_offset": avg_right_offset,
        "left_edge": left_edge,
        "right_edge": right_edge,
        "x_span": x_span,
        "y_span": y_span,
        "RA": ra,
        "DEC": dec,
        "zip_file": zip_name
    })

# ======== Export výsledkov =========
df = pd.DataFrame(results)
df.to_csv("warp_results_original_cutouts.csv", index=False)

print("Saved warp_results_original_cutouts.csv")

14. Výpočet warpu zo sigma‑clipped cutoutov a archivácia FITS verzií

Táto časť pipeline spracúva všetky FITS cutouty po odstránení pozadia (sigma‑clipped verzia). Pre každý objekt sa vykoná PCA zarovnanie disku, výpočet warpu na ľavej a pravej strane, meranie rozsahov v rotovanom rámci a extrakcia RA/DEC z katalógu.
Zároveň sa vytvára ZIP balík obsahujúci tri verzie FITS súboru (pôvodný, cutout, sigma‑clipped).
Výsledky sa ukladajú do tabuľky warp_results_sigma_cutouts.csv.

In [ ]:
import numpy as np
import pandas as pd
import os
import shutil
import zipfile
from astropy.io import fits
from tqdm import tqdm

# ======== Nastavenia =========
input_dir    = "FITS_CUTOUTS_NEW"      # cutouty pred sigma-clippingom
original_dir = "FITS_CUTOUTS"          # pôvodné cutouty
sigma_dir    = "CUTOUT_NEW_FINALS_"    # sigma-clipped cutouty
BIN_SIZE     = 2                       # veľkosť binu v rotovanom rámci

# Katalóg s RA/DEC a názvami súborov
catalog = pd.read_csv("modified_dataset.csv")

# Zoznam FITS súborov na spracovanie (sigma-clipped verzia)
all_fits_files = sorted([f for f in os.listdir(sigma_dir) if f.endswith(".fits")])
fits_paths = [os.path.join(sigma_dir, f) for f in all_fits_files]

def rotate_points(x, y, ang):
    """Rotácia bodov (x, y) o uhol ang (radiány)."""
    cos_t = np.cos(ang)
    sin_t = np.sin(ang)
    xr = x * cos_t - y * sin_t
    yr = x * sin_t + y * cos_t
    return xr, yr

def calculate_flux_center(dat, x_value):
    """
    Výpočet fluxom váženého stredu profilu v osi Y pre daný X_bin.
    Používa jednoduchú numerickú integráciu.
    """
    subset = dat[dat['X_bin'] == x_value]
    if len(subset) < 2:
        return None

    subset = subset.sort_values('Y_rot')
    y = subset['Y_rot'].values
    flux = subset['value'].values

    dy = np.diff(y)
    numerator   = np.sum(flux[1:] * y[1:] * dy)
    denominator = np.sum(flux[1:] * dy)

    if denominator == 0:
        return None

    return numerator / denominator

results = []

# Priečinky pre ZIP archívy
output_zip_folder = "output_zips"
os.makedirs(output_zip_folder, exist_ok=True)

tmp_zip_folder = "tmp_zip_folder"
os.makedirs(tmp_zip_folder, exist_ok=True)

# ======== Hlavná slučka =========
for file in tqdm(fits_paths, desc="Processing galaxies"):

    file_base = os.path.basename(file).split('.')[0]
    sigma_val = np.nan  # placeholder, ak by si chcel doplniť sigma hodnoty

    try:
        # ===== FITS =====
        with fits.open(file, memmap=True) as hdu:
            img = np.array(hdu[0].data, dtype=np.float64)

        # Výber nenan a nenulových pixelov
        mask = (~np.isnan(img)) & (img != 0)
        y_idx, x_idx = np.where(mask)

        if len(x_idx) == 0:
            continue

        values = img[y_idx, x_idx]
        if len(values) == 0:
            continue

        data = pd.DataFrame({
            'X': x_idx,
            'Y': y_idx,
            'value': values
        })

        weights = np.clip(values, 0, None)

        # ===== CORE (top 1 % pixelov) =====
        threshold = np.percentile(values, 99)
        core = data[data['value'] >= threshold]
        if len(core) == 0:
            continue

        center_x = np.average(core['X'], weights=core['value'])
        center_y = np.average(core['Y'], weights=core['value'])

        # Posun do stredu galaxie
        data['X'] -= center_x
        data['Y'] -= center_y

        # ===== PCA =====
        coords = data[['X', 'Y']].values
        mean = np.average(coords, axis=0, weights=weights)
        coords_centered = coords - mean

        cov = np.cov(coords_centered.T, aweights=weights)
        eigvals, eigvecs = np.linalg.eigh(cov)

        major_axis = eigvecs[:, np.argmax(eigvals)]
        angle = np.arctan2(major_axis[1], major_axis[0])

        # ===== ROTÁCIA =====
        x_rot, y_rot = rotate_points(data['X'].values, data['Y'].values, -angle)

        data_rot = data.copy()
        data_rot['X_rot'] = x_rot
        data_rot['Y_rot'] = y_rot

        # ===== Opätovné zarovnanie jadra =====
        flux_threshold = np.percentile(data_rot['value'], 95)
        core_pixels = data_rot[data_rot['value'] >= flux_threshold]
        if len(core_pixels) == 0:
            continue

        core_y = np.average(core_pixels['Y_rot'], weights=core_pixels['value'])
        data_rot['Y_rot'] -= core_y

        # ===== BINNING =====
        data_rot['X_bin'] = (np.floor(data_rot['X_rot'] / BIN_SIZE) * BIN_SIZE).astype(int)
        data_rot['Y_bin'] = (np.floor(data_rot['Y_rot'] / BIN_SIZE) * BIN_SIZE).astype(int)

        binned = data_rot.groupby(['X_bin', 'Y_bin']).agg({
            'X_rot': 'mean',
            'Y_rot': 'mean',
            'value': 'mean'
        }).reset_index()

        binned_left  = binned[binned['X_bin'] <= 0]
        binned_right = binned[binned['X_bin'] >= 0]

        # ===== Výpočet warpu =====
        warp_left = []
        warp_right = []

        for x in sorted(binned_left['X_bin'].unique()):
            yc = calculate_flux_center(binned_left, x)
            if yc is not None:
                warp_left.append((x, yc))

        for x in sorted(binned_right['X_bin'].unique()):
            yc = calculate_flux_center(binned_right, x)
            if yc is not None:
                warp_right.append((x, yc))

        warp_left  = pd.DataFrame(warp_left,  columns=["X", "Y_center"])
        warp_right = pd.DataFrame(warp_right, columns=["X", "Y_center"])

        warp_combined = pd.concat([warp_left, warp_right]).sort_values('X')

        # Zarovnanie centrálnej oblasti na Y≈0
        if len(warp_combined) > 0:
            center_region = warp_combined[np.abs(warp_combined['X']) < 10]
            if len(center_region) > 0:
                y0 = np.average(center_region['Y_center'])
                warp_combined['Y_center'] -= y0

        # ===== Parametre warpu =====
        left_integral  = np.trapezoid(warp_left['Y_center'],  warp_left['X'])  if len(warp_left)  > 0 else np.nan
        right_integral = np.trapezoid(warp_right['Y_center'], warp_right['X']) if len(warp_right) > 0 else np.nan

        avg_left_offset  = warp_left['Y_center'].mean()  if len(warp_left)  > 0 else np.nan
        avg_right_offset = warp_right['Y_center'].mean() if len(warp_right) > 0 else np.nan

        left_edge  = warp_left['Y_center'].iloc[0]   if len(warp_left)  > 0 else np.nan
        right_edge = warp_right['Y_center'].iloc[-1] if len(warp_right) > 0 else np.nan

        x_span = data_rot['X_rot'].max() - data_rot['X_rot'].min()
        y_span = data_rot['Y_rot'].max() - data_rot['Y_rot'].min()

        # ===== RA/DEC =====
        row = catalog[catalog['jpg_loc_gz_arcsinh_vis_only'].str.split('.').str[0] == file_base]
        ra  = row.iloc[0]['right_ascension'] if len(row) > 0 else np.nan
        dec = row.iloc[0]['declination']     if len(row) > 0 else np.nan

        # ===== ZIP archív =====
        for key, path in {"original": original_dir, "cutout": input_dir, "sigma": sigma_dir}.items():
            src = os.path.join(path, file_base + ".fits")
            if os.path.exists(src):
                shutil.copy(src, os.path.join(tmp_zip_folder, f"{file_base}_{key}.fits"))

        zip_name = os.path.join(output_zip_folder, f"{file_base}_all_versions.zip")

        with zipfile.ZipFile(zip_name, 'w') as zf:
            for f in os.listdir(tmp_zip_folder):
                if f.startswith(file_base):
                    zf.write(os.path.join(tmp_zip_folder, f), arcname=f)

        # Reset dočasného priečinka
        shutil.rmtree(tmp_zip_folder)
        os.makedirs(tmp_zip_folder, exist_ok=True)

        # ===== Uloženie výsledkov =====
        results.append({
            "galaxy_id": file_base,
            "left_side_integral": left_integral,
            "avg_left_offset": avg_left_offset,
            "right_side_integral": right_integral,
            "avg_right_offset": avg_right_offset,
            "left_edge": left_edge,
            "right_edge": right_edge,
            "x_span": x_span,
            "y_span": y_span,
            "RA": ra,
            "DEC": dec,
            "sigma": sigma_val,
            "zip_file": zip_name
        })

    except Exception as e:
        print(f"Error processing {file_base}: {e}")
        continue

# ===== Export výsledkov =====
df = pd.DataFrame(results)
df.to_csv("warp_results_sigma_cutouts.csv", index=False)

print("Saved warp_results_sigma_cutouts.csv")

15. Priradenie sigma‑threshold hodnoty ku galaxii podľa veľkosti cutoutu

Táto časť pipeline dopĺňa do tabuľky warp_results_sigma_cutouts.csv stĺpec sigma, ktorý určuje, aký sigma‑threshold bol použitý pri odstraňovaní pozadia pre daný FITS cutout. Sigma sa určuje podľa veľkosti cutoutu: malé galaxie používajú prah 3σ, väčšie 5σ.
Výsledná tabuľka sa aktualizuje priamo v pôvodnom CSV.

In [ ]:
import os
import numpy as np
import pandas as pd
from astropy.io import fits
from tqdm import tqdm

# ======== Nastavenia =========
fits_dir = "FITS_CUTOUTS_NEW"        # cutouty po aplikovaní masiek
csv_file = "warp_results_sigma_cutouts.csv"

# ======== Načítanie výsledkov warpu =========
df = pd.read_csv(csv_file)

# ======== Zoznam FITS súborov =========
fits_files = sorted([f for f in os.listdir(fits_dir) if f.endswith(".fits")])
fits_dict = {f.split('.')[0]: f for f in fits_files}

# ======== Funkcia na určenie sigma thresholdu =========
def get_sigma(file_path):
    """
    Určí sigma threshold podľa veľkosti cutoutu.
    Menšie galaxie → 3σ
    Väčšie galaxie → 5σ
    """
    if not os.path.exists(file_path):
        return np.nan

    try:
        with fits.open(file_path, memmap=True) as hdu:
            img = np.array(hdu[0].data, dtype=np.float64)

            if img.size == 0:
                return np.nan

            gal_size = max(img.shape)

            if gal_size <= 60:
                return 3
            else:
                return 5

    except Exception:
        return np.nan

# ======== Priradenie sigma hodnoty pre každú galaxiu =========
sigmas = []

for gid in tqdm(df['galaxy_id'], desc="Assigning sigma"):

    base_name = str(gid).split('.')[0]
    matched_file = fits_dict.get(base_name)

    if matched_file:
        file_path = os.path.join(fits_dir, matched_file)
        sigma_val = get_sigma(file_path)
    else:
        sigma_val = np.nan

    sigmas.append(sigma_val)

# ======== Pridanie stĺpca do tabuľky =========
df['sigma'] = sigmas

# ======== Uloženie aktualizovaného CSV =========
df.to_csv(csv_file, index=False)

print(f"Sigma updated in {csv_file}")